In [34]:
from dolfinx import log, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
import numpy as np
from dolfinx.io.gmsh import read_from_msh
import ufl
import gmsh
from mpi4py import MPI
from dolfinx import fem, mesh, plot
def facets_to_nodes(domain, facet_ids):
    fdim = domain.topology.dim - 1
    domain.topology.create_connectivity(fdim, 0)
    facet_to_vertices = domain.topology.connectivity(fdim, 0)
    all_nodes = []
    for f in facet_ids:
        all_nodes.extend(facet_to_vertices.links(f))
    return np.unique(np.array(all_nodes, dtype=np.int32))

def FEM_solve(load_start, load_end, load_steps) :
    gmsh.initialize()
    gdim = 2
    model = gmsh.model.occ
    L_x = 1.0
    L_y = 1.0
    R_hole = 0.1
    mesh_size = 0.02
    # 2. Create rectangle
    rect = model.addRectangle(0.0, 0.0, 0.0, L_x, L_y)

    # 3. Create circle
    circle = model.addDisk(L_x/2, L_x/2, 0.0, R_hole, R_hole)

    # 4. Cut the hole
    domain_with_hole, _ = model.cut([(gdim, rect)], [(gdim, circle)])
    gmsh.model.occ.synchronize()

    # 5. Physical group
    gmsh.model.addPhysicalGroup(gdim, [rect], 1)
    gmsh.model.setPhysicalName(gdim, 1, "domain")
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", mesh_size)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", mesh_size)
    gmsh.model.mesh.generate(gdim)
    # 6. Mesh generation
    gmsh.model.mesh.generate(gdim)

    # 7. Write to file
    gmsh.write("mesh.msh")
    gmsh.finalize()

    # 8. Read into DOLFINx
    mesh_ = read_from_msh("mesh.msh", MPI.COMM_WORLD, 0, gdim)
    domain = mesh_.mesh
    V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim,)))


    # -

    # We create two python functions for determining the facets to apply boundary conditions to


    # +
    def left(x):
        return np.isclose(x[0], 0)


    def right(x):
        return np.isclose(x[0], L_x)


    def top(x):
        return np.isclose(x[1], L_y)


    def bottom(x):
        return np.isclose(x[1], 0)
    def corner(x):
        return np.isclose(x[0], 0.0) & np.isclose(x[1], 0.0) 
    def gegenrotation(x) :
        return np.isclose(x[0], L_x) & np.isclose(x[1], L_y) 

    fdim = domain.topology.dim - 1
    left_facets = mesh.locate_entities_boundary(domain, fdim, left)
    right_facets = mesh.locate_entities_boundary(domain, fdim, right)
    top_facets = mesh.locate_entities_boundary(domain, fdim, top)
    bottom_facets = mesh.locate_entities_boundary(domain, fdim, bottom)
    corner_facets = mesh.locate_entities_boundary(domain, fdim, corner)
    gegenrotation_facets = mesh.locate_entities_boundary(domain, fdim, gegenrotation)
    # -

    # Next, we create a  marker based on these two functions

    # Concatenate and sort the arrays based on facet indices. Left facets marked with 1, right facets with two

    marked_facets = np.hstack([left_facets, bottom_facets, right_facets, top_facets, corner_facets, gegenrotation_facets])
    marked_values = np.hstack([np.full_like(left_facets, 1), np.full_like(bottom_facets, 2), np.full_like(right_facets, 3), np.full_like(top_facets, 4), np.full_like(corner_facets, 5), np.full_like(gegenrotation_facets, 6)])
    sorted_facets = np.argsort(marked_facets)
    facet_tag = mesh.meshtags(
        domain, fdim, marked_facets[sorted_facets], marked_values[sorted_facets]
    )

    # We then create a function for supplying the boundary condition on the left side, which is fixed.

    u_bc = np.array((0,) * domain.geometry.dim, dtype=default_scalar_type)

    # To apply the boundary condition, we identity the dofs located on the facets marked by the `MeshTag`.
    zero = fem.Constant(domain, default_scalar_type(0.0))

    # Locate DOFs (note the use of collapsed spaces)
    left_dofs = fem.locate_dofs_topological(V.sub(0), facet_tag.dim, facet_tag.find(1))
    bottom_dofs = fem.locate_dofs_topological(V.sub(1), facet_tag.dim, facet_tag.find(2))
    bottom_dofs_xy = fem.locate_dofs_topological(V, facet_tag.dim, facet_tag.find(2))
    right_dofs = fem.locate_dofs_topological(V.sub(0), facet_tag.dim, facet_tag.find(3))
    top_dofs = fem.locate_dofs_topological(V.sub(1), facet_tag.dim, facet_tag.find(4))
    corner_dofs = fem.locate_dofs_topological(V, facet_tag.dim, facet_tag.find(5))
    gegen_dofs = fem.locate_dofs_topological(V.sub(1), facet_tag.dim, facet_tag.find(6))
    bcs = [
        fem.dirichletbc(u_bc, corner_dofs, V),
        fem.dirichletbc(zero, gegen_dofs, V.sub(1))
        # fem.dirichletbc(zero, left_dofs, V.sub(0)), 
        # fem.dirichletbc(zero, bottom_dofs, V.sub(1)),
        # fem.dirichletbc(u_bc, bottom_dofs_xy, V),
        # fem.dirichletbc(control_ux, right_dofs, V.sub(0)),
        # fem.dirichletbc(control_uy, top_dofs, V.sub(1))
        ]

    # Next, we define the body force on the reference configuration (`B`), and nominal (first Piola-Kirchhoff) traction (`T`).

    B = fem.Constant(domain, default_scalar_type((0, 0)))
    T_right = fem.Constant(domain, default_scalar_type((0.0, 0.0)))
    T_top = fem.Constant(domain, default_scalar_type((0.0, load_start * 1.1)))
    T_left = fem.Constant(domain, default_scalar_type((0.0, 0.0)))
    # Define the test and solution functions on the space $V$

    v = ufl.TestFunction(V)
    u = fem.Function(V)

    # Define kinematic quantities used in the problem

    # +
    # Spatial dimension
    d = len(u)

    # Identity tensor
    I = ufl.variable(ufl.Identity(3))
    def grad_3D(u):
        return ufl.as_matrix([[u[0].dx(0), u[0].dx(1), 0], 
                            [u[1].dx(0), u[1].dx(1), 0], 
                            [0, 0, 0]])
    F = ufl.variable(I + grad_3D(u))
    # Deformation gradient
    # F = ufl.variable(I + ufl.grad(u))

    # Right Cauchy-Green tensor
    C = ufl.variable(F.T * F)

    # Invariants of deformation tensors
    Ic = ufl.variable(ufl.tr(C))
    I2 = ufl.variable(0.5 * (ufl.tr(C)**2 - ufl.tr(C * C)))
    J = ufl.variable(ufl.det(F))
    # -

    # Define the elasticity model via a stored strain energy density function $\psi$,
    # and create the expression for the first Piola-Kirchhoff stress:

    # Elasticity parameters

    # Stored strain energy density (compressible neo-Hookean model)

    # psi = (mu / 2) * (Ic - 3) - mu * ufl.ln(J) + (lmbda / 2) * (ufl.ln(J)) ** 2

    psi = 0.5 * (J**(-2/3) * Ic - 3) + 1.5 * (J - 1)**2 # neohookean
    # psi = 0.5 * (J**(-2/3) * Ic - 3) + 1.0 * (J**(-4/3) * I2 - 3) + 1.5 * (J - 1)**2 # mr

    # psi = 0.5 * (J**(-2/3) * Ic - 3) + (J**(-2/3) * Ic - 3)**2 + (J**(-4/3) * I2 - 3) + 1.5 * (J - 1)**2 #isihara
    # psi = 0.5 * (J**(-2/3) *Ic - 3) + ufl.ln(J**(-4/3) * I2/3) + 1.5 * (J - 1)**2 Gent Thomas
    # Hyper-elasticity

    P = ufl.diff(psi, F)

    metadata = {"quadrature_degree": 4}
    ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tag, metadata=metadata)
    dx = ufl.Measure("dx", domain=domain, metadata=metadata)

    # Define the residual of the equation (we want to find u such that residual(u) = 0)

    residual = (
        ufl.inner(grad_3D(v), P) * dx - ufl.inner(v, B) * dx - ufl.inner(v, T_right) * ds(3) + ufl.inner(v, T_left) * ds(1)
    )

    # As the varitional form is non-linear and written on residual form,
    # we use the non-linear problem class from DOLFINx to set up required structures to use a Newton solver.

    petsc_options = {
        "snes_type": "newtonls",
        "snes_linesearch_type": "none",
        "snes_monitor": None,
        "snes_atol": 1e-10,
        "snes_rtol": 1e-10,
        "snes_stol": 1e-10,
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
    }
    problem = NonlinearProblem(
        residual,
        u,
        bcs=bcs,
        petsc_options=petsc_options,
        petsc_options_prefix="hyperelasticity",
        
    )
    # load_per_step = (load_end - load_start)/load_steps
    load_steps = np.linspace(load_start, load_end, load_steps)
    u_steps = {}

    mesh_pos = domain.geometry.x
    fdim = domain.topology.dim - 1 
    domain.topology.create_connectivity(fdim, domain.topology.dim)
    cells = domain.topology.connectivity(domain.topology.dim, 0).array.reshape(-1, 3)
    num_nodes = domain.geometry.x.shape[0]
    node_type = np.zeros(num_nodes, dtype=int)
    node_type[facets_to_nodes(domain, left_facets)] = 1
    node_type[facets_to_nodes(domain, bottom_facets)] = 2
    node_type[facets_to_nodes(domain, right_facets)] = 3
    node_type[facets_to_nodes(domain, top_facets)] = 4

    for i, n in enumerate(load_steps):
        T_right.value[0] = n
        T_left.value[1] = n
        # T_top.value[0] = n * 1.5
        problem.solve()

        u_array = u.x.array[:].reshape(mesh_pos.shape[0], -1)
        u_steps[i] = u_array.copy()


    return dict(mesh_pos = mesh_pos, cells = cells, u = u_steps, node_type = node_type, load_steps = load_steps)


In [36]:
data = FEM_solve(1.0, 1.0, 20)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 5 (Ellipse)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Line)
Info    : [ 70%] Meshing curve 8 (Line)
Info    : [ 90%] Meshing curve 9 (Line)
Info    : Done meshing 1D (Wall 0.000581134s, CPU 0.001023s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0633198s, CPU 0.064376s)
Info    : 3002 nodes 6009 elements
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0555251s, CPU 0.051624s)
Info    : 3002 nodes 6009 elements
Info    : Writing 'mesh.msh'...
Info    : Done writing 'mesh.msh'
Info    : Reading 'mesh.msh'...
Info    : 11 entities
Info    : 3002 nodes
Info    : 5772 elements
Info    : Done reading 'mesh.msh'
  0 SNES Function norm 1.989974874213e-01
  1 SNES Function norm 1.458660673127e+40
  0 SNES Function norm 1.458660673127e+40
  1 SNES Function norm 4.321957550007e+39
 

In [37]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# --- 1. Setup Dummy Data (Simulating FEM Output) ---
# Create a simple 2x2 rectangular mesh with 4 nodes and 2 triangular elements
# Node coordinates (Undeformed mesh_pos)
mesh_pos = data["mesh_pos"]

# Element connectivity (cells: indices of nodes forming each triangle)
cells = data["cells"]

# Displacement components (ux and uy) at each node
# This simulates a simple shear/tensile deformation
percent_noise = 0.0
ux = data["u"][len(data["u"].keys()) - 1][:, 0]
ux[(data["node_type"] != 1)] += np.random.normal(0, percent_noise * ux.max(), ux.shape)[(data["node_type"] != 1)]
uy = data["u"][len(data["u"].keys()) - 1][:, 1]
uy[(data["node_type"] != 2)] += np.random.normal(0, percent_noise * uy.max(), uy.shape)[(data["node_type"] != 2)]

# Combine components into the full displacement vector u
u = np.column_stack((ux, uy))
# u[data["node_type"] == 0] = u[data["node_type"] == 0] + np.random.normal(0, 0.05 * np.mean(u, axis = 0), u.shape)[data["node_type"] == 0]
# Calculate the deformed coordinates (world_pos)
world_pos = mesh_pos[:, :2] + u

# --- 2. Initialize Triangulation Objects ---
# We need the x and y coordinates from the undeformed mesh
x = mesh_pos[:, 0]
y = mesh_pos[:, 1]

# Create the Matplotlib Triangulation object
# This object stores the connectivity (cells) and coordinates (x, y)
triangulation = tri.Triangulation(x, y, cells)

# --- 3. Plotting ---

# Set up the figure with 1 row and 3 columns for the plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Finite Element Visualization (Undeformed vs. Deformed)', fontsize=16)

# --- Subplot 1: Plotting UX (Horizontal Displacement) ---
ax1 = axes[0]
# tripcolor uses the triangulation to color the triangles based on the nodal value
# `facecolors` uses the average of the three nodal values per triangle for coloring
tpc1 = ax1.tripcolor(triangulation, ux, cmap='viridis', edgecolors='k', linewidth=0.5)
fig.colorbar(tpc1, ax=ax1, label='$u_x$ Displacement')
# ax1.scatter(mesh_pos[data["node_type"] == 5, 0], mesh_pos[node_type == 5, 1])
ax1.set_title('Color Plot: $u_x$ (Horizontal Displacement)')
ax1.set_xlabel('X Position')
ax1.set_ylabel('Y Position')
ax1.set_aspect('equal')

# --- Subplot 2: Plotting UY (Vertical Displacement) ---
ax2 = axes[1]
tpc2 = ax2.tripcolor(triangulation, uy, cmap='magma', edgecolors='k', linewidth=0.01)
fig.colorbar(tpc2, ax=ax2, label='$u_y$ Displacement')

ax2.set_title('Color Plot: $u_y$ (Vertical Displacement)')
ax2.set_xlabel('X Position')
ax2.set_aspect('equal')

# --- Subplot 3: Plotting Deformed Domain ---
ax3 = axes[2]

# Plot the outline of the UNDEFORMED mesh for reference (dashed gray)
ax3.triplot(triangulation, 'r-', alpha=0.5, linewidth=0.5, label='Undeformed Mesh')

# Plot the DEFORMED mesh. We must manually create a new triangulation
# object using the deformed coordinates (world_pos) but the SAME connectivity (cells).
x_def = world_pos[:, 0]
y_def = world_pos[:, 1]
tri_def = tri.Triangulation(x_def, y_def, cells)

# Plot the deformed mesh (solid blue lines)
ax3.triplot(tri_def, 'b-', linewidth=0.5, label='Deformed Mesh')

ax3.set_title('Deformed Domain')
ax3.set_xlabel('X Position')
ax3.legend()
ax3.set_aspect('equal')

# Adjust layout to prevent overlaps
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

ValueError: scale < 0